In [5]:
import os, json, time
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool, StructuredTool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")
print("환경 설정 완료")

환경 설정 완료


In [6]:
# 서울 날씨 알려줘? -> LLM -> 답변을 못할 것 같으면 tool_calls -> 개발자가 만든 함수 -> result -> 서울은 맑아요.
def get_weather(city):
    weather_db = {
        'seoul': '맑음, 22도, 습도 45%',
        'tokyo': '흐림, 19도, 습도 70%',
        'new york': '비, 15도, 습도 80%',
    }
    return weather_db.get(city, f"{city} : 정보 없음")

In [7]:
tool_calls = {'name': 'get_weather', 'args' : {'city' : 'seoul'}}
result = get_weather(**tool_calls['args'])

In [8]:
result

'맑음, 22도, 습도 45%'

In [9]:
# LLM이 호출 가능한 tool이다 라는 것을 알려줌
@tool
def get_weather(city):
    """날씨를 조회하는 함수입니다."""
    weather_db = {
        'seoul': '맑음, 22도, 습도 45%',
        'tokyo': '흐림, 19도, 습도 70%',
        'new york': '비, 15도, 습도 80%',
    }
    return weather_db.get(city, f"{city} : 정보 없음")

In [10]:
get_weather.name

'get_weather'

In [11]:
get_weather.description

'날씨를 조회하는 함수입니다.'

In [12]:
llm_with_tools = llm.bind_tools([get_weather, get_exchange_rate])

NameError: name 'get_exchange_rate' is not defined

In [ ]:
result =llm_with_tools.invoke('서울 날씨 어때?')

In [ ]:
if result.tool_calls:
    tc = result.tool_calls[0]
    print(f"tool : {tc['name']}")

    result_final = get_weather.invoke(tc['args'])
    print(f"result_final : {result_final}")

In [ ]:
@tool
def get_exchange_rate(from_currency: str, to_currency: str) -> str:
    """두 통화 간 환율을 조회하는 함수입니다."""
    
    currency_db = {
        ("USD", "KRW"): "1 USD = 1350 KRW",
        ("KRW", "USD"): "1 KRW = 0.00074 USD",
        ("USD", "JPY"): "1 USD = 150 JPY",
        ("EUR", "KRW"): "1 EUR = 1450 KRW",
    }
    
    key = (from_currency.upper(), to_currency.upper())
    
    return currency_db.get(
        key,
        f"{from_currency} -> {to_currency} 환율 정보 없음"
    )

In [ ]:
result=llm_with_tools.invoke('원달러 환율 알려줘')

In [ ]:
result

In [ ]:
# StructuredTool.from_function

In [ ]:
def search_etf_func(category, min_return=0, max_expense=1.0):
    etfs = [
        {"name": "KODEX 200", "cat": "국내주식", "ret": 8.5, "exp": 0.15},
        {"name": "KODEX S&P500TR", "cat": "해외주식", "ret": 25.3, "exp": 0.05},
        {"name": "TIGER 나스닥100", "cat": "해외주식", "ret": 30.2, "exp": 0.07},
        {"name": "ACE 미국배당다우존스", "cat": "배당", "ret": 12.1, "exp": 0.01},
    ]

    result = [e for e in etfs if e['cat'] == category and e['ret'] >= min_return and e['exp'] <= max_expense]
    return json.dumps(result, ensure_ascii=False) if result else '조건에 맞는 ETF 없음'

In [ ]:
from pydantic import BaseModel, Field

class ETFSearchInput(BaseModel):
    category: str = Field(description='ETF 카테고리')
    min_return: float = Field(default=0, ge=0, le=100, description='최소 수익률')
    max_expense: float = Field(default=0, ge=0, le=5, description='최대 운용보수')

In [ ]:
search_etf = StructuredTool.from_function(
    func = search_etf_func, name='search_etf',
    description='조건에 맞는 상품을 검색합니다.',
    args_schema=ETFSearchInput,
)
search_etf

In [ ]:
search_etf.name, search_etf.args

In [ ]:
llm_etf = llm.bind_tools([search_etf])
queries = ['해외주식 ETF 추천해줘', '수수료 낮은 배당 ETF 찾아줘']
for q in queries:
    resp = llm_etf.invoke(q)
    if resp.tool_calls:
        tc = resp.tool_calls[0]
        result = search_etf.invoke(tc['args'])
        print(f"Q : {q}")
        print(f"A : {result}")

In [ ]:
class EmployeeSearchInput(BaseModel):
    department: str = Field(description="부서명 (예: 개발, 마케팅, 인사)")
    min_salary: int = Field(default=0, ge=0, description="최소 연봉")

def search_employee_func(department, min_salary=0):
    employees = [
        {"name": "김철수", "dept": "개발", "salary": 5000},
        {"name": "이영희", "dept": "마케팅", "salary": 4200},
        {"name": "박민수", "dept": "개발", "salary": 6500},
        {"name": "최지은", "dept": "인사", "salary": 3800},
    ]

    result = [
        e for e in employees
        if e["dept"] == department and e["salary"] >= min_salary
    ]

    return json.dumps(result, ensure_ascii=False) if result else "조건에 맞는 직원 없음"

search_employee = StructuredTool.from_function(
    func=search_employee_func,
    name="search_employee",
    description="부서와 최소 연봉 조건으로 직원을 검색합니다.",
    args_schema=EmployeeSearchInput,
)

llm_employee = llm.bind_tools([search_employee])

queries = [
    "개발 부서 직원 찾아줘",
    "연봉 5000 이상 개발자 보여줘",
]

for q in queries:
    resp = llm_employee.invoke(q)
    
    if resp.tool_calls:
        tc = resp.tool_calls[0]
        result = search_employee.invoke(tc["args"])
        
        print(f"Q : {q}")
        print(f"A : {result}")

In [ ]:
all_tools = [get_weather, get_exchange_rate, search_etf]
llm_multi = llm.bind_tools(all_tools)

In [ ]:
resp = llm_multi.invoke('서울과 도쿄 날씨 비교하고 환율도 알려줘')
resp

In [ ]:
tool_map = {t.name: t for t in all_tools}
for tc in resp.tool_calls:
    result = tool_map[tc['name']].invoke(tc['args'])
    print(f"{tc['name']} {tc['args']} -> {result}")

In [ ]:
@tool
def calculator(expression: str) -> str:
    """수식을 계산하는 계산기 함수입니다. 예: '15*24', '100/4'"""
    try:
        result = eval(expression)
        return str(result)
    except Exception:
        return "계산 오류"

llm_with_tools = llm.bind_tools([
    get_weather,
    get_exchange_rate,
    calculator
])

def run_query(query: str):
    resp = llm_with_tools.invoke(query)

    print(f"\nQ: {query}")

    if resp.tool_calls:
        for tc in resp.tool_calls:
            name = tc["name"]
            args = tc["args"]

            if name == "get_weather":
                result = get_weather.invoke(args)
            elif name == "get_exchange_rate":
                result = get_exchange_rate.invoke(args)
            elif name == "calculator":
                result = calculator.invoke(args)
            else:
                result = "Unknown tool"

            print(f"[{name}] → {result}")
    else:
        print("A:", resp.content)

queries = [
    "서울 날씨 알려줘",
    "원달러 환율 알려줘",
    "15*24 계산해줘",
    "서울 날씨 알려주고 15*24 계산해줘"
]

for q in queries:
    run_query(q)

In [ ]:
def tool_loop(query, tools, max_turns=5):
    tool_map = {t.name: t for t in all_tools}
    llm_t = llm.bind_tools(tools)

    messages = [HumanMessage(content=query)]
    
    for turn in range(max_turns):
        response = llm_t.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for tc in response.tool_calls:
            tool_obj = tool_map[tc['name']]
            result = tool_obj.invoke(tc['args']) if tool_obj else '도구 없음'
            messages.append(
                ToolMessage(content=str(result), tool_call_id=tc['id'])
            )
    return '최대 초과'